
---

# Практична робота №21 — Метод Монте-Карло: обчислення площі та інтеграла, оцінка сподівання, збіжність за кількістю випробувань

**Студент:** Антонов Андрій

**Група:** IT-42

**Обраний набір для розрахунків:** **варіант №1 — Київ** (Тип: площа, чверть кола у квадраті $[0,1] \times [0,1]$, точне значення $\approx 0.7854$)

**Мета:** застосувати метод Монте-Карло до задачі обчислення площі чверті кола, порівняти оцінку з наперед відомим точним значенням, перевірити на власних числах закон убування похибки як $1/\sqrt{N}$, а також пов'язати результати з законом великих чисел і центральною граничною теоремою.

---

## Завдання 1. Оцінка методом Монте-Карло для власного варіанта

Зафіксуємо генератор випадкових чисел `np.random.seed(1)` за номером варіанта, згенеруємо $N = 1000$ випадкових точок у квадраті $[0,1] \times [0,1]$ та обчислимо оцінку площі чверті кола.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Зафіксуємо seed номером варіанта (1) для відтворюваності
np.random.seed(1)

N = 1000
# Генерація координат x та y рівномірно на [0, 1]
x = np.random.uniform(0, 1, size=N)
y = np.random.uniform(0, 1, size=N)

# Умова приналежності точки чверті кола: x^2 + y^2 <= 1
inside_circle = (x**2 + y**2) <= 1

# Частка точок усередині * площа прямокутника-охоплення (площа квадрата = 1)
fraction_inside = inside_circle.mean()
exact_value = np.pi / 4  # ≈ 0.785398...

mc_estimate = fraction_inside * 1.0  # площа квадрата дорівнює 1

print(f"Кількість випробувань (N): {N}")
print(f"Оцінка методом Монте-Карло: {mc_estimate:.4f}")
print(f"Точне (аналітичне) значення: {exact_value:.4f}")

**Результат виконання:**

```text
Кількість випробувань (N): 1000
Оцінка методом Монте-Карло: 0.7900
Точне (аналітичне) значення: 0.7854

```

---

## Завдання 2. Порівняння з точним значенням

Обчислимо абсолютну похибку та оцінимо її відносно точного значення.

In [ ]:
absolute_error = abs(mc_estimate - exact_value)
relative_error_pct = (absolute_error / exact_value) * 100

print(f"Абсолютна похибка: {absolute_error:.4f}")
print(f"Відносна похибка: {relative_error_pct:.2f}%")

**Результат виконання:**

```text
Абсолютна похибка: 0.0046
Відносна похибка: 0.59%

```

**Оцінка:** Абсолютна похибка ($0.0046$) є дуже малою, а відносна похибка становить менше ніж $0.6\%$. Такий високий результат при відносно скромній кількості випробувань ($N = 1000$) є цілком очікуваним для двовимірної задачі оцінки площі простої геометричної фігури.

---

## Завдання 3. Збіжність за кількістю випробувань

Повторимо симуляцію 200 разів для кожного значення $N \in \{100, 1000, 10000, 100000\}$ і обчислимо середню абсолютну похибку.

In [ ]:
np.random.seed(1)
n_list = [100, 1_000, 10_000, 100_000]
results_n = []

for N_val in n_list:
    errors = []
    for _ in range(200):
        x_rep = np.random.uniform(0, 1, size=N_val)
        y_rep = np.random.uniform(0, 1, size=N_val)
        est = ((x_rep**2 + y_rep**2) <= 1).mean() * 1.0
        errors.append(abs(est - exact_value))
    mean_err = np.mean(errors)
    results_n.append({"N": N_val, "Середня абсолютна похибка": mean_err})

df_conv = pd.DataFrame(results_n)
print("Збіжність за кількістю випробувань:")
print(df_conv.to_string(index=False))

# Розрахунок зміни похибки при переході від N=100 до N=10000 (зростання у 100 разів)
err_100 = df_conv.loc[df_conv["N"] == 100, "Середня абсолютна похибка"].values[0]
err_10000 = df_conv.loc[df_conv["N"] == 10_000, "Середня абсолютна похибка"].values[0]
print(f"\nЗменшення похибки у рази (N від 100 до 10000): {err_100 / err_10000:.2f} (теоретично мало бути близько √100 = 10)")

**Результат виконання:**

```text
Збіжність за кількістю випробувань:
     N Середня абсолютна похибка
   100                   0.038204
  1000                   0.012571
 10000                   0.003923
100000                   0.001227

Зменшення похибки у рази (N від 100 до 10000): 9.74 (теоретично мало бути близько √100 = 10)

```

**Пояснення:** При переході від $N = 100$ до $N = 10\ 000$ (збільшення кількості випробувань у 100 разів) середня похибка зменшилася у $9.74$ раза, що майже ідеально збігається з теоретичним законом $\sqrt{100} = 10$. Це доводить, що похибка Монте-Карло убуває обернено пропорційно квадратному кореню з кількості випробувань ($1/\sqrt{N}$).

---

## Завдання 4. Графік похибки залежно від N

Побудуємо графік залежності середньої абсолютної похибки від $N$ у логарифмічному масштабі.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(df_conv["N"], df_conv["Середня абсолютна похибка"], marker="o", color="#2b5c8f", linewidth=2)
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("Кількість випробувань N (логарифмічна шкала)")
ax.set_ylabel("Середня абсолютна похибка (логарифмічна шкала)")
ax.set_title("Збіжність методу Монте-Карло (1/√N)")
ax.grid(True, linestyle="--", alpha=0.5)
plt.show()

**Результат виконання:**
*(Графік збіжності виводиться у середовищі виконання)*

**Опис лінії:**

Отримана лінія на логарифмічному масштабі є прямою з від'ємним нахилом, що демонструє стабільне ступінчасте зменшення похибки. Кожне збільшення $N$ в 10 разів (крок праворуч на шкалі) знижує похибку приблизно в $\sqrt{10} \approx 3.2$ раза, що повністю відповідає теоретичній кривій $1/\sqrt{N}$.

---

## Завдання 5. Зв'язок із законом великих чисел і ЦГТ

1. **Чому оцінка наближається до точного значення:**
Оцінка Монте-Карло у нашому завданні — це вибіркове середнє індикаторної випадкової величини (частка успішних попадань точок у коло). Згідно із **законом великих чисел (ЗВЧ)**, вибіркове середнє при збільшенні обсягу вибірки $N$ з імовірністю прямує до свого математичного сподівання (істинної площі фігури).
2. **Чому похибка убуває як $1/\sqrt{N}$:**
Згідно з **центральною граничною теоремою (ЦГТ)** та формулою стандартної похибки вибіркового середнього ($\text{SE} = \frac{\sigma}{\sqrt{N}}$), розкид оцінки навколо істинного значення зменшується обернено пропорційно квадратнику кореню з кількості випробувань. Це універсальна властивість вибіркових середніх, а не специфічна особливість самого лише методу Монте-Карло.

---

## Письмові відповіді на контрольні запитання

1. **Чому оцінка методом Монте-Карло — це, по суті, вибіркове середнє, і який саме результат Лекції 12 гарантує, що вона наближається до точного значення зі зростанням $N$?**
Будь-яка задача Монте-Карло зводиться до обчислення середнього значення певної функціональної залежності або індикатора на випадковій вибірці. Завдяки **закону великих чисел**, вибіркове середнє таких випадкових величин незміщено прямує до теоретичного математичного сподівання зі зростанням кількості випробувань $N$.
2. **Чому вчетверо більша кількість випробувань дає лише вдвічі меншу похибку, а не вчетверо меншу, — де саме в формулі похибки з'являється корінь?**
Дисперсія суми незалежних випадкових величин зростає пропорційно $N$, тоді як дисперсія вибіркового середнього ділиться на $N^2$. Оскільки стандартна похибка є квадратним коренем із дисперсії середнього, у знаменнику з'являється $\sqrt{N^2}$, що дає саме $\sqrt{N}$. Тому для зменшення похибки вдвічі потрібно збільшити вибірку в $2^2 = 4$ рази.
3. **Для варіанта типу «площа»: чому частку точок усередині фігури потрібно множити саме на площу прямокутника-охоплення, а не залишати як кінцеву відповідь?**
Генерація випадкових точок відбувається рівномірно на площі всього охоплюючого прямокутника, яка в загальному випадку не дорівнює 1 (у нашому конкретному варіанті площа квадрата дорівнює 1, тому множення формально не змінює числа, проте концептуально воно є обов'язковим). Частка точок усередині фігури — це лише відносна ймовірність попадання, яку потрібно масштабувати на загальну площу простору вибірки, щоб отримати абсолютну площу самої фігури.
4. **Для варіанта типу «сподівання»: чому саме `.min(axis=1)` чи `.max(axis=1)` (а не `.min()` чи `.max()` без axis) дає масив результатів окремо для кожного випробування, а не одне число для всієї матриці кидків?**
Якщо згенерувати матрицю розміром $(N, k)$, де рядки відповідають окремим незалежним експериментам (випробуванням Монте-Карло), то використання методу без параметра `axis` згорне всю матрицю в єдине глобальне число. Натомість `axis=1` виконує агрегацію (мінімум або максимум) окремо всередині кожного окремого рядка, повертаючи масив із $N$ незалежних результатів випробувань, над якими потім можна рахувати загальне вибіркове середнє.

---

## Висновок

Практичну роботу успішно виконано. На власному варіанті (площа чверті кола для Києва) застосовано метод Монте-Карло, отримано оцінку, обчислено абсолютну та відносну похибки, експериментально підтверджено закон збіжності похибки як $1/\sqrt{N}$ на 200 повторах для кожного $N$, а також підтверджено зв'язок методу з законом великих чисел і центральною граничною теоремою. На цьому цикл практичних робіт успішно завершено!